# 06 — Clear Results: CFD to ROM Metrics

## Section 1 — Flow Rate Analysis

This notebook extracts CFD-derived flow-rate quantities that can later be transferred into 0D and 1D reduced-order models. It deliberately excludes pressure drop, resistance, wall shear stress, energy loss, and calibration.

Case: `openfoam/segment_test_V2`

Outputs are written to `output/06_clear_results/`.


In [ ]:
from pathlib import Path
import ast
import csv
import json
import math
import os
import re

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(value):
        print(value)

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        case_data = candidate / "openfoam" / "segment_test_V2" / "data"
        if case_data.exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate repository root containing openfoam/segment_test_V2/data. "
        f"Current working directory is: {Path.cwd()}"
    )

PROJECT_ROOT = find_project_root()
CASE_DIR = PROJECT_ROOT / "openfoam" / "segment_test_V2"
DATA_DIR = CASE_DIR / "data"
OUTPUT_DIR = PROJECT_ROOT / "output" / "06_clear_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"CFD data directory: {DATA_DIR}")

SLICES = ["inlet", "midslice", "outlet"]
SEARCH_TERMS = [
    "integrated", "integrate variables", "integration", "inlet", "midslice",
    "outlet", "flow rate", "flow_rate", "velocity integration", "velocity", "area"
]

# Slice normals recovered from the existing CFD post-processing workflow for this case.
# They are used to compute Q = integral_A (u dot n) dA from ParaView vector integrations.
SLICE_NORMALS = {
    "inlet": np.array([0.5103037804570406, -0.8596550644734353, -0.024149985019174598]),
    "midslice": np.array([0.6324414356546973, -0.7702129056188429, 0.08240091313331853]),
    "outlet": np.array([0.4278871251313104, -0.9017025116588478, 0.06200958486385685]),
}

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.frameon": False,
})

def read_csv_rows(path):
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))

def write_csv(path, rows):
    if not rows:
        return
    columns = []
    for row in rows:
        for key in row:
            if key not in columns:
                columns.append(key)
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)

def markdown_table(rows, columns=None, float_format=".6g"):
    if not rows:
        return "(no rows)"
    if columns is None:
        columns = list(rows[0].keys())
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for row in rows:
        values = []
        for col in columns:
            value = row.get(col, "")
            if isinstance(value, (float, np.floating)) and math.isfinite(float(value)):
                values.append(format(float(value), float_format))
            else:
                values.append(str(value))
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)

def show_table(rows, columns=None, title=None, float_format=".6g"):
    text = markdown_table(rows, columns=columns, float_format=float_format)
    if title:
        text = f"**{title}**\n\n" + text
    display(Markdown(text))

def safe_read_text(path, limit=20000):
    try:
        with path.open("r", encoding="utf-8", errors="ignore") as handle:
            return handle.read(limit)
    except Exception:
        return ""

def csv_header(path):
    try:
        with path.open(newline="") as handle:
            reader = csv.reader(handle)
            return next(reader, [])
    except Exception:
        return []

def detected_quantities(path, text, header):
    labels = []
    joined = " ".join(header).lower() + " " + text[:5000].lower()
    if all(col in header for col in ["U:0", "U:1", "U:2"]):
        labels.append("velocity components U")
    if "Area" in header or "area" in joined:
        labels.append("area")
    if "p" in header or "pressure" in joined:
        labels.append("pressure/kinematic pressure")
    if "q" in joined or "flow" in joined:
        labels.append("flow rate")
    if "normal" in joined:
        labels.append("slice normal")
    if "cell type" in joined:
        labels.append("cell type")
    if "velocity" in joined and "velocity components U" not in labels:
        labels.append("velocity")
    return "; ".join(labels) if labels else "candidate post-processing data"

def infer_purpose(path, header):
    name = path.name.lower()
    parts = " ".join(path.parts).lower()
    slice_name = next((s for s in SLICES if s in name or s in parts), "slice")
    has_u = all(col in header for col in ["U:0", "U:1", "U:2"])
    has_area = "Area" in header
    if has_u and has_area:
        return f"ParaView Integrate Variables export for {slice_name}"
    if has_u:
        return f"raw velocity samples for {slice_name}"
    if "flow" in name:
        return "flow-rate summary or calculation"
    if "integration" in name or "integrated" in name:
        return "integrated slice quantities"
    if any(s in name for s in SLICES):
        return f"{slice_name} data"
    return "candidate CFD/ROM post-processing artifact"


## Step 1 — Data Discovery

The notebook searches the repository for files whose names, columns, or readable text indicate integrated slice quantities, ParaView Integrate Variables exports, inlet/midslice/outlet data, flow-rate calculations, or velocity integrations.


In [ ]:
def discover_candidate_files(root):
    candidates = []
    skip_dirs = {".git", ".ipynb_checkpoints", "__pycache__"}
    extensions = {".csv", ".dat", ".txt", ".xy", ".pvsm", ".ipynb", ".foam", ".OpenFOAM", ""}
    for path in root.rglob("*"):
        if not path.is_file():
            continue
        if any(part in skip_dirs for part in path.parts):
            continue
        if path.suffix not in extensions:
            continue
        text = safe_read_text(path)
        header = csv_header(path) if path.suffix == ".csv" else []
        haystack = f"{path.name} {' '.join(path.parts)} {' '.join(header)} {text[:5000]}".lower()
        if not any(term in haystack for term in SEARCH_TERMS):
            continue
        candidates.append({
            "file": path.name,
            "location": str(path.relative_to(root).parent),
            "purpose": infer_purpose(path, header),
            "detected quantities": detected_quantities(path, text, header),
            "path": str(path.relative_to(root)),
        })
    return sorted(candidates, key=lambda row: ("openfoam/segment_test_V2" not in row["path"], row["path"]))

def flow_section_used_files(root):
    used_specs = [
        ("inlet_integration_variable.csv", "flow-rate integration"),
        ("midslice_integration_variable.csv", "flow-rate integration"),
        ("outlet_integration_variable.csv", "flow-rate integration"),
        ("inlet_integration.csv", "velocity-profile diagnostics"),
        ("midslice_integration.csv", "velocity-profile diagnostics"),
        ("outlet_integration.csv", "velocity-profile diagnostics"),
    ]
    rows = []
    for file_name, used_for in used_specs:
        path = DATA_DIR / file_name
        header = csv_header(path) if path.exists() else []
        rows.append({
            "file": file_name,
            "location": str(path.relative_to(root).parent) if path.exists() else str(DATA_DIR.relative_to(root)),
            "purpose": infer_purpose(path, header) if path.exists() else "required CFD slice export not found",
            "used_for": used_for,
        })
    return rows

inventory_rows = discover_candidate_files(PROJECT_ROOT)
write_csv(OUTPUT_DIR / "data_discovery_inventory.csv", inventory_rows)
show_table(
    flow_section_used_files(PROJECT_ROOT),
    columns=["file", "location", "purpose", "used_for"],
    title="Files Used in This Flow-Rate Section",
)
print(f"Full inventory saved to {OUTPUT_DIR / 'data_discovery_inventory.csv'}")


## Step 2 — Extract Flow Rate

For each slice, the required CFD quantity is

`Q = integral_A (u dot n) dA`.

The ParaView Integrate Variables exports provide the integrated velocity vector components. The slice normal then projects those components onto the through-plane direction. Areas are converted from m² to mm² for reporting.


In [ ]:
def find_slice_file(slice_name, integrated=True):
    suffix = "integration_variable.csv" if integrated else "integration.csv"
    direct = DATA_DIR / f"{slice_name}_{suffix}"
    if direct.exists():
        return direct

    candidates = []
    for row in inventory_rows:
        path = PROJECT_ROOT / row["path"]
        header = csv_header(path)
        if not all(col in header for col in ["U:0", "U:1", "U:2"]):
            continue
        if slice_name not in row["path"].lower():
            continue
        has_area = "Area" in header
        if integrated and not has_area:
            continue
        if not integrated and has_area:
            continue
        score = 0
        if str(path).startswith(str(CASE_DIR / "data")):
            score += 10
        if "integration_variable" in path.name.lower():
            score += 5
        if "integration.csv" in path.name.lower():
            score += 3
        candidates.append((score, path))
    if not candidates:
        checked = direct.relative_to(PROJECT_ROOT) if direct.is_absolute() else direct
        raise FileNotFoundError(
            f"No {'integrated velocity/area' if integrated else 'raw velocity sample'} CSV found for {slice_name}. "
            f"First checked: {checked}"
        )
    return sorted(candidates, key=lambda item: item[0], reverse=True)[0][1]

integrated_files = {name: find_slice_file(name, integrated=True) for name in SLICES}
raw_velocity_files = {name: find_slice_file(name, integrated=False) for name in SLICES}

flow_records = []
for name in SLICES:
    row = read_csv_rows(integrated_files[name])[0]
    u_integral = np.array([float(row["U:0"]), float(row["U:1"]), float(row["U:2"])])
    normal = SLICE_NORMALS[name] / np.linalg.norm(SLICE_NORMALS[name])
    area_m2 = float(row["Area"])
    q_m3_s = float(np.dot(u_integral, normal))
    flow_records.append({
        "Slice": name,
        "Area [mm²]": area_m2 * 1e6,
        "Flow rate [m³/s]": q_m3_s,
        "Flow rate [mL/s]": q_m3_s * 1e6,
        "normal_x": float(normal[0]),
        "normal_y": float(normal[1]),
        "normal_z": float(normal[2]),
        "source_file": str(integrated_files[name].relative_to(PROJECT_ROOT)),
    })

write_csv(OUTPUT_DIR / "flow_rate_summary.csv", flow_records)
show_table(flow_records, columns=["Slice", "Area [mm²]", "Flow rate [m³/s]", "Flow rate [mL/s]"], title="CFD Flow Rate Summary")


## Step 3 — Mass Conservation Analysis

Mass conservation is checked by comparing midslice and outlet flow rates against the inlet reference flow rate. Small relative differences indicate that the integrated flow is conserved across the sampled cross-sections.


In [ ]:
q_by_slice = {row["Slice"]: row["Flow rate [m³/s]"] for row in flow_records}
q_ref = q_by_slice["inlet"]

mass_rows = []
for name in SLICES:
    q = q_by_slice[name]
    rel_error = 100.0 * (q - q_ref) / q_ref if q_ref else float("nan")
    mass_rows.append({
        "Slice": name,
        "Q [m³/s]": q,
        "Q [mL/s]": q * 1e6,
        "Relative Error vs Inlet [%]": rel_error,
        "Absolute Relative Error [%]": abs(rel_error),
    })

write_csv(OUTPUT_DIR / "mass_conservation_summary.csv", mass_rows)
show_table(mass_rows, columns=["Slice", "Q [m³/s]", "Relative Error vs Inlet [%]"], title="Mass Conservation Check")

mid_err = abs(next(row["Relative Error vs Inlet [%]"] for row in mass_rows if row["Slice"] == "midslice"))
out_err = abs(next(row["Relative Error vs Inlet [%]"] for row in mass_rows if row["Slice"] == "outlet"))
threshold = 1.0
status = "satisfied" if max(mid_err, out_err) <= threshold else "not fully satisfied"
display(Markdown(
    f"**Interpretation:** Mass conservation is **{status}** using a {threshold:.1f}% tolerance. "
    f"The inlet-to-midslice difference is {mid_err:.3f}% and the inlet-to-outlet difference is {out_err:.3f}%."
))

fig, ax = plt.subplots(figsize=(6.6, 4.2))
x = np.arange(len(SLICES))
y = [q_by_slice[name] * 1e6 for name in SLICES]
ax.plot(x, y, marker="o", linewidth=2.2)
ax.set_xticks(x, SLICES)
ax.set_ylabel("Flow rate [mL/s]")
ax.set_title("CFD Flow Rate Across Slices")
ax.margins(x=0.08, y=0.18)
for xi, yi in zip(x, y):
    ax.annotate(f"{yi:.4f}", (xi, yi), textcoords="offset points", xytext=(0, 8), ha="center")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "flow_rate_inlet_midslice_outlet.png", bbox_inches="tight")
plt.show()


## Step 4 — Flow Information Transferability

**What CFD provides**

- Full velocity field.
- Local velocity variations.
- Velocity profile across each slice.

**What is transferred to 1D**

- Flow rate `Q`.
- Average velocity `Q/A`.

**What is transferred to 0D**

- Lumped flow rate.

Flow rate is preserved in both 1D and 0D models. Velocity profiles are not preserved.


In [ ]:
transfer_rows = []
for row in flow_records:
    q = row["Flow rate [m³/s]"]
    area_m2 = row["Area [mm²]"] * 1e-6
    transfer_rows.append({
        "Slice": row["Slice"],
        "Transferable Q [mL/s]": q * 1e6,
        "Transferable average velocity Q/A [m/s]": q / area_m2,
        "0D transfer": "lumped Q",
        "1D transfer": "Q and Q/A",
        "Not transferred": "velocity profile",
    })
write_csv(OUTPUT_DIR / "flow_transferability_summary.csv", transfer_rows)
show_table(transfer_rows, title="Flow Quantities Transferable to ROMs")


## Step 5 — Velocity Distribution Diagnostics

If raw slice velocity samples are available, this section computes local velocity-magnitude diagnostics and plots histograms for inlet, midslice, and outlet. These distributions show that CFD contains local velocity information even when the integrated flow rate is nearly identical across slices. That local profile information is lost during 1D/0D reduction.


In [ ]:
velocity_records = []
velocity_samples = {}
missing_raw = []
for name in SLICES:
    path = raw_velocity_files[name]
    if path is None:
        missing_raw.append(name)
        continue
    rows = read_csv_rows(path)
    vectors = np.array([[float(row["U:0"]), float(row["U:1"]), float(row["U:2"])] for row in rows])
    magnitudes = np.linalg.norm(vectors, axis=1)
    velocity_samples[name] = magnitudes
    velocity_records.append({
        "Slice": name,
        "n samples": len(magnitudes),
        "mean velocity [m/s]": float(np.mean(magnitudes)),
        "maximum velocity [m/s]": float(np.max(magnitudes)),
        "standard deviation [m/s]": float(np.std(magnitudes, ddof=1)) if len(magnitudes) > 1 else 0.0,
        "source_file": str(path.relative_to(PROJECT_ROOT)),
    })

if missing_raw:
    display(Markdown(f"Raw velocity samples were not found for: {', '.join(missing_raw)}."))

write_csv(OUTPUT_DIR / "velocity_distribution_summary.csv", velocity_records)
show_table(velocity_records, title="Velocity Distribution Diagnostics")

if velocity_samples:
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    for name in SLICES:
        if name not in velocity_samples:
            continue
        ax.hist(velocity_samples[name], bins=36, density=True, alpha=0.42, label=name)
    ax.set_xlabel("Velocity magnitude [m/s]")
    ax.set_ylabel("Density")
    ax.set_title("CFD Slice Velocity Magnitude Distributions")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "velocity_magnitude_histograms.png", bbox_inches="tight")
    plt.show()


## Step 5B — Velocity Profile Non-Uniformity Analysis

Scientific question: does a conserved flow rate imply identical flow behaviour across the vessel?

This section tests whether nearly conserved `Q` across inlet, midslice, and outlet also means the local velocity field is the same. It is explicitly framed as the first example of information loss during CFD → 1D → 0D reduction.


In [ ]:
# Load all available raw velocity sample exports and compute non-uniformity metrics.
profile_records = []
profile_samples = {}
profile_points = {}

for name in SLICES:
    path = raw_velocity_files.get(name)
    if path is None:
        continue
    rows = read_csv_rows(path)
    if not rows:
        continue

    vectors = np.array([[float(row["U:0"]), float(row["U:1"]), float(row["U:2"])] for row in rows])
    magnitudes = np.linalg.norm(vectors, axis=1)
    mean_u = float(np.mean(magnitudes))
    std_u = float(np.std(magnitudes, ddof=1)) if len(magnitudes) > 1 else 0.0
    cv_u = std_u / mean_u if mean_u else float("nan")

    profile_samples[name] = magnitudes
    profile_records.append({
        "Slice": name,
        "Mean Velocity [m/s]": mean_u,
        "Median Velocity [m/s]": float(np.median(magnitudes)),
        "Max Velocity [m/s]": float(np.max(magnitudes)),
        "Std Velocity [m/s]": std_u,
        "CV [-]": cv_u,
        "NonUniformity [-]": cv_u,
        "n samples": len(magnitudes),
        "source_file": str(path.relative_to(PROJECT_ROOT)),
    })

    header = rows[0].keys()
    if {"Points:0", "Points:1", "Points:2"}.issubset(header):
        profile_points[name] = np.array([[float(row["Points:0"]), float(row["Points:1"]), float(row["Points:2"])] for row in rows])

write_csv(OUTPUT_DIR / "velocity_profile_nonuniformity_summary.csv", profile_records)
show_table(
    profile_records,
    columns=["Slice", "Mean Velocity [m/s]", "Median Velocity [m/s]", "Max Velocity [m/s]", "Std Velocity [m/s]", "CV [-]"],
    title="Velocity Profile Non-Uniformity Summary",
)


In [ ]:
# Overlay velocity magnitude histograms for inlet, midslice, and outlet.
if profile_samples:
    fig, ax = plt.subplots(figsize=(7.4, 4.8))
    for name in SLICES:
        if name not in profile_samples:
            continue
        ax.hist(profile_samples[name], bins=40, density=True, alpha=0.42, label=name)
    ax.set_xlabel("Velocity magnitude [m/s]")
    ax.set_ylabel("Density")
    ax.set_title("Velocity Magnitude Distributions Across CFD Slices")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "velocity_profile_histograms_overlay.png", bbox_inches="tight")
    plt.show()

    means = {row["Slice"]: row["Mean Velocity [m/s]"] for row in profile_records}
    max_vals = {row["Slice"]: row["Max Velocity [m/s]"] for row in profile_records}
    cvs = {row["Slice"]: row["CV [-]"] for row in profile_records}
    accelerated = max(means, key=means.get)
    longest_tail = max(max_vals, key=max_vals.get)
    most_variable = max(cvs, key=cvs.get)

    display(Markdown(
        "**Histogram interpretation:** The distributions are not identical. "
        f"The highest mean velocity occurs at the **{accelerated}**, indicating local acceleration there. "
        f"The longest high-velocity tail occurs at the **{longest_tail}**, and the strongest relative spread occurs at the **{most_variable}**. "
        "This shows that conserved flow rate does not imply identical local flow behaviour."
    ))


In [ ]:
# Boxplots expose spread, skew, and high-velocity tails more directly than the mean alone.
if profile_samples:
    available = [name for name in SLICES if name in profile_samples]
    fig, ax = plt.subplots(figsize=(6.8, 4.8))
    boxplot_values = [profile_samples[name] for name in available]
    try:
        ax.boxplot(
            boxplot_values,
            tick_labels=available,
            showfliers=True,
            flierprops={"marker": ".", "markersize": 2.5, "alpha": 0.35},
        )
    except TypeError:
        fallback_tick_kwargs = {"labels": available}
        ax.boxplot(
            boxplot_values,
            **fallback_tick_kwargs,
            showfliers=True,
            flierprops={"marker": ".", "markersize": 2.5, "alpha": 0.35},
        )
    ax.set_ylabel("Velocity magnitude [m/s]")
    ax.set_title("Velocity Magnitude Spread by Slice")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "velocity_profile_boxplots.png", bbox_inches="tight")
    plt.show()

    stds = {row["Slice"]: row["Std Velocity [m/s]"] for row in profile_records}
    max_vals = {row["Slice"]: row["Max Velocity [m/s]"] for row in profile_records}
    largest_spread = max(stds, key=stds.get)
    highest_velocity = max(max_vals, key=max_vals.get)
    display(Markdown(
        "**Boxplot interpretation:** "
        f"The **{largest_spread}** has the largest absolute velocity spread, while the **{highest_velocity}** contains the highest velocities. "
        "The changing spread across slices is consistent with vessel geometry influencing the local velocity distribution, even while integrated flow is conserved."
    ))


In [ ]:
# Non-uniformity metric: Std(U) / Mean(U).
if profile_records:
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    names = [row["Slice"] for row in profile_records]
    values = [row["NonUniformity [-]"] for row in profile_records]
    bars = ax.bar(names, values)
    ax.set_ylabel("NonUniformity = Std(U) / Mean(U) [-]")
    ax.set_title("Velocity Non-Uniformity by Slice")
    ax.set_ylim(0, max(values) * 1.22 if values else 1)
    for bar, value in zip(bars, values):
        ax.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value), textcoords="offset points", xytext=(0, 6), ha="center")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "velocity_nonuniformity_bar.png", bbox_inches="tight")
    plt.show()

    most_nonuniform = names[int(np.argmax(values))]
    display(Markdown(
        f"**Non-uniformity interpretation:** Higher values indicate stronger spatial velocity variation. "
        f"The largest non-uniformity is observed at the **{most_nonuniform}**, so this slice loses the most local velocity-structure information when reduced to a single average velocity or lumped flow rate."
    ))


In [ ]:
# Optional spatial velocity maps when ParaView exports contain Points:0, Points:1, Points:2.
def plane_coordinates(points, normal):
    normal = normal / np.linalg.norm(normal)
    reference = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(reference, normal)) > 0.9:
        reference = np.array([0.0, 1.0, 0.0])
    e1 = np.cross(normal, reference)
    e1 = e1 / np.linalg.norm(e1)
    e2 = np.cross(normal, e1)
    centered = points - np.mean(points, axis=0)
    return centered @ e1, centered @ e2

if len(profile_points) == len(profile_samples) and profile_points:
    display(Markdown(
        "**Coordinate-aware analysis:** `Points:0`, `Points:1`, and `Points:2` were found, so cross-sectional velocity maps are generated. "
        "These maps are preferred scientifically because they show where the non-uniformity occurs."
    ))
    fig, axes = plt.subplots(1, len(SLICES), figsize=(14, 4.2), constrained_layout=True)
    if len(SLICES) == 1:
        axes = [axes]
    vmax = max(float(np.max(profile_samples[name])) for name in profile_points)
    last = None
    for ax, name in zip(axes, SLICES):
        pts = profile_points[name]
        x, y = plane_coordinates(pts, SLICE_NORMALS[name])
        last = ax.scatter(x * 1000, y * 1000, c=profile_samples[name], s=9, cmap="viridis", vmin=0, vmax=vmax)
        ax.set_title(name)
        ax.set_xlabel("local x [mm]")
        ax.set_aspect("equal", adjustable="box")
    axes[0].set_ylabel("local y [mm]")
    fig.colorbar(last, ax=axes, label="Velocity magnitude [m/s]", shrink=0.82)
    fig.savefig(OUTPUT_DIR / "velocity_spatial_maps.png", bbox_inches="tight")
    plt.show()
else:
    display(Markdown(
        "**Coordinate-aware analysis:** The current raw exports do not include `Points:0`, `Points:1`, and `Points:2`, so true cross-sectional velocity maps cannot be reconstructed here. "
        "If future ParaView exports include point coordinates, this cell will automatically generate spatial velocity maps, which are stronger than histograms because they show where non-uniformity occurs."
    ))


### Step 5B Discussion — First Information-Loss Example

**Key observation:** Flow rate is conserved across the vessel, but the velocity field is not uniform.

Two slices can have nearly identical flow rates while exhibiting very different velocity distributions. This demonstrates the first important information loss during CFD → 1D → 0D reduction.

**CFD preserves**

- Local velocity structure.
- Velocity gradients.
- Asymmetric flow patterns.
- Curvature effects.

**1D preserves**

- Flow rate.
- Average velocity.

**0D preserves**

- Only lumped flow rate.

Therefore, matching flow rate between CFD and reduced-order models does not imply recovery of the underlying 3D flow field.


In [ ]:
final_5b_lines = [
    "✓ Flow rate is conserved.",
    "✓ Flow rate can be transferred to 1D and 0D models.",
    "✓ Velocity distributions are non-uniform.",
    "✓ Velocity profile information is lost during model reduction.",
    "✓ Agreement in flow rate alone is insufficient to claim agreement between CFD and reduced-order models.",
]
write_csv(OUTPUT_DIR / "velocity_profile_nonuniformity_conclusion.csv", [{"Result": line} for line in final_5b_lines])
display(Markdown("**Final Result Statement**\n\n" + "\n".join(final_5b_lines)))


## Step 6 — Final Flow Rate Conclusions

This final cell summarizes only the flow-rate and velocity-distribution findings from Section 1.


In [ ]:
conclusion_rows = [
    {
        "Question": "Is flow conserved?",
        "Conclusion": f"Yes, within {max(mid_err, out_err):.3f}% across inlet, midslice, and outlet for this extracted CFD dataset.",
    },
    {
        "Question": "Which quantity is transferable to ROMs?",
        "Conclusion": "Flow rate Q is the primary conserved quantity; 1D models also use average velocity Q/A.",
    },
    {
        "Question": "Which information is lost?",
        "Conclusion": "Local velocity distributions and velocity profiles are collapsed in 1D and absent in 0D.",
    },
]
write_csv(OUTPUT_DIR / "flow_rate_conclusions.csv", conclusion_rows)
show_table(conclusion_rows, title="Final Flow Rate Conclusions")

print(f"Figures and tables written to: {OUTPUT_DIR}")


# Section 2 — Pressure Drop and CFD Resistance

This section uses the CFD pressure field to extract the pressure loss across the vessel segment and compute an equivalent hydraulic resistance for later calibration of 0D/1D reduced-order models.

Only pressure drop and resistance are analyzed here. Wall shear stress, energy loss, transient effects, and full model calibration are intentionally left out.


## Step 2.1 — Required CFD Data

This section automatically uses the integrated slice files:

- `openfoam/segment_test_V2/data/inlet_integration_variable.csv`
- `openfoam/segment_test_V2/data/midslice_integration_variable.csv`
- `openfoam/segment_test_V2/data/outlet_integration_variable.csv`

Each file is expected to contain `Area`, `p`, `U:0`, `U:1`, and `U:2`.

OpenFOAM pressure `p` is kinematic pressure for this case. It is converted to physical pressure using:

`p_Pa = rho * p_kinematic`

with `rho = 1060 kg/m^3`. Pressure in Pa is converted to mmHg using:

`1 mmHg = 133.322 Pa`.


In [ ]:
RHO_BLOOD = 1060.0
PA_PER_MMHG = 133.322

pressure_files = {name: find_slice_file(name, integrated=True) for name in SLICES}
required_pressure_columns = {"Area", "p", "U:0", "U:1", "U:2"}
missing_pressure_columns = {}
for name, path in pressure_files.items():
    header = set(csv_header(path))
    missing = sorted(required_pressure_columns - header)
    if missing:
        missing_pressure_columns[name] = missing

if missing_pressure_columns:
    details = "; ".join(f"{name}: {', '.join(cols)}" for name, cols in missing_pressure_columns.items())
    raise ValueError(f"Required pressure columns are missing from integrated slice files: {details}")

show_table(
    [
        {
            "Slice": name,
            "file": pressure_files[name].name,
            "location": str(pressure_files[name].relative_to(PROJECT_ROOT).parent),
        }
        for name in SLICES
    ],
    title="Integrated CFD Files Used for Pressure Analysis",
)


## Step 2.2 — Average Pressure at Each Slice

For each slice:

`p_avg_kinematic = integrated_p / area`

`p_avg_Pa = rho * p_avg_kinematic`

`p_avg_mmHg = p_avg_Pa / 133.322`


In [ ]:
pressure_slice_records = []
pressure_by_slice = {}
for name in SLICES:
    row = read_csv_rows(pressure_files[name])[0]
    area_m2 = float(row["Area"])
    integrated_p = float(row["p"])
    p_avg_kinematic = integrated_p / area_m2
    p_avg_pa = RHO_BLOOD * p_avg_kinematic
    p_avg_mmhg = p_avg_pa / PA_PER_MMHG
    pressure_by_slice[name] = {
        "area_m2": area_m2,
        "p_avg_kinematic": p_avg_kinematic,
        "p_avg_pa": p_avg_pa,
        "p_avg_mmhg": p_avg_mmhg,
    }
    pressure_slice_records.append({
        "Slice": name,
        "Area [mm²]": area_m2 * 1e6,
        "Average pressure [Pa]": p_avg_pa,
        "Average pressure [mmHg]": p_avg_mmhg,
    })

write_csv(OUTPUT_DIR / "pressure_slice_summary.csv", pressure_slice_records)
show_table(pressure_slice_records, title="CFD Average Pressure at Each Slice")


## Step 2.3 — Pressure Drop Along Vessel

Pressure drops are computed as upstream average pressure minus downstream average pressure for inlet→midslice, midslice→outlet, and inlet→outlet.


In [ ]:
pressure_segments = [
    ("inlet→midslice", "inlet", "midslice"),
    ("midslice→outlet", "midslice", "outlet"),
    ("inlet→outlet", "inlet", "outlet"),
]
pressure_drop_records = []
for segment, upstream, downstream in pressure_segments:
    delta_pa = pressure_by_slice[upstream]["p_avg_pa"] - pressure_by_slice[downstream]["p_avg_pa"]
    pressure_drop_records.append({
        "Segment": segment,
        "Delta P [Pa]": delta_pa,
        "Delta P [mmHg]": delta_pa / PA_PER_MMHG,
    })

write_csv(OUTPUT_DIR / "pressure_drop_summary.csv", pressure_drop_records)
show_table(pressure_drop_records, title="CFD Pressure Drop Between Slices")

slice_labels = SLICES
pressure_values_mmhg = [pressure_by_slice[name]["p_avg_mmhg"] for name in slice_labels]
fig, ax = plt.subplots(figsize=(6.6, 4.2))
ax.plot(slice_labels, pressure_values_mmhg, marker="o")
ax.set_xlabel("Slice")
ax.set_ylabel("Average pressure [mmHg]")
ax.set_title("CFD Average Pressure Along Vessel")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "pressure_profile_mmHg.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7.2, 4.2))
segments = [row["Segment"] for row in pressure_drop_records]
deltas = [row["Delta P [mmHg]"] for row in pressure_drop_records]
bars = ax.bar(segments, deltas)
ax.set_ylabel("Pressure drop [mmHg]")
ax.set_title("CFD Pressure Drop Between Slices")
for bar, value in zip(bars, deltas):
    ax.annotate(f"{value:.3g}", (bar.get_x() + bar.get_width() / 2, value), textcoords="offset points", xytext=(0, 6 if value >= 0 else -14), ha="center")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "pressure_drop_bar.png", bbox_inches="tight")
plt.show()


## Step 2.4 — Pressure Validation

The expected qualitative behaviour is a monotonic pressure decrease from inlet to midslice to outlet.


In [ ]:
p_inlet = pressure_by_slice["inlet"]["p_avg_pa"]
p_midslice = pressure_by_slice["midslice"]["p_avg_pa"]
p_outlet = pressure_by_slice["outlet"]["p_avg_pa"]
pressure_decreases_monotonically = p_inlet > p_midslice > p_outlet

if pressure_decreases_monotonically:
    print("Pressure decreases monotonically from inlet to outlet.")
else:
    print(
        "Warning: pressure does not decrease monotonically from inlet to outlet. "
        "This may indicate incorrect slice orientation, pressure reference issues, or non-physical extraction."
    )


## Step 2.5 — CFD Hydraulic Resistance

Using the Section 1 inlet flow rate:

`R_CFD = Delta P_inlet_outlet / Q_inlet`

Resistance is reported in `Pa·s/m³` and `mmHg·s/mL`, with:

`R_mmHg_s_mL = R_Pa_s_m3 / (133.322 * 1e6)`.


In [ ]:
q_inlet_m3_s = q_by_slice["inlet"]
delta_p_inlet_outlet_pa = next(row["Delta P [Pa]"] for row in pressure_drop_records if row["Segment"] == "inlet→outlet")
delta_p_inlet_outlet_mmhg = delta_p_inlet_outlet_pa / PA_PER_MMHG
r_cfd_pa_s_m3 = delta_p_inlet_outlet_pa / q_inlet_m3_s
r_cfd_mmhg_s_ml = r_cfd_pa_s_m3 / (PA_PER_MMHG * 1e6)

resistance_rows = [{
    "Quantity": "CFD-equivalent hydraulic resistance",
    "Q_inlet [mL/s]": q_inlet_m3_s * 1e6,
    "Delta P inlet-outlet [mmHg]": delta_p_inlet_outlet_mmhg,
    "R_CFD [Pa·s/m³]": r_cfd_pa_s_m3,
    "R_CFD [mmHg·s/mL]": r_cfd_mmhg_s_ml,
}]
write_csv(OUTPUT_DIR / "cfd_resistance_summary.csv", resistance_rows)
show_table(resistance_rows, title="CFD Hydraulic Resistance Summary")


## Step 2.6 — Interpretation for ROM Transferability

Pressure drop and hydraulic resistance are directly transferable to 0D/1D models.

**0D** uses the pressure-flow relation directly, with resistance represented as a lumped parameter.

**1D** distributes resistance along the vessel using radius, length, and friction assumptions.

**CFD** captures the real pressure loss caused by geometry, radius variation, curvature, and local velocity effects.

Matching `R_CFD` allows the reduced-order model to reproduce the global pressure-flow response, but it does not recover local 3D pressure or velocity fields.


## Step 2.7 — Final Section 2 Conclusion

This table summarizes the pressure-drop and resistance findings only.


In [ ]:
pressure_conclusion_rows = [
    {
        "Question": "Does pressure decrease along the vessel?",
        "Conclusion": "Yes, pressure decreases monotonically from inlet to outlet." if pressure_decreases_monotonically else "No, the extracted pressure does not decrease monotonically from inlet to outlet.",
    },
    {
        "Question": "What is the total pressure drop?",
        "Conclusion": f"The inlet-to-outlet pressure drop is {delta_p_inlet_outlet_mmhg:.6g} mmHg ({delta_p_inlet_outlet_pa:.6g} Pa).",
    },
    {
        "Question": "What is the CFD-equivalent resistance?",
        "Conclusion": f"R_CFD = {r_cfd_pa_s_m3:.6g} Pa·s/m³ = {r_cfd_mmhg_s_ml:.6g} mmHg·s/mL.",
    },
    {
        "Question": "Why is this important for 0D/1D?",
        "Conclusion": "It gives the global pressure-flow response that reduced-order models can match through a lumped or distributed resistance.",
    },
    {
        "Question": "What information is still lost?",
        "Conclusion": "Matching resistance does not recover local 3D pressure fields, velocity fields, or spatial flow structures.",
    },
]
write_csv(OUTPUT_DIR / "pressure_resistance_conclusions.csv", pressure_conclusion_rows)
show_table(pressure_conclusion_rows, title="Final Pressure and Resistance Conclusions")

print(f"Pressure and resistance outputs written to: {OUTPUT_DIR}")


# Section 3 — Resistance Analysis: From CFD to Reduced-Order Models

## Scientific Goal

This section asks: what information is lost when replacing a real 3D CFD vessel by a reduced-order representation?

The purpose is not yet to compare CFD against existing 0D or 1D simulation outputs. This section only compares:

`CFD ↔ analytical Poiseuille theory`

The workflow is:

1. Extract the effective hydraulic resistance from CFD.
2. Compute theoretical Poiseuille resistance from geometry.
3. Quantify the difference.
4. Interpret the difference as information loss caused by reducing the real 3D vessel to a simplified hydraulic representation.
5. Define calibration quantities that can later be used in 0D and 1D models.


## Step 3.1 — Load CFD Resistance

The CFD resistance from Section 2 is loaded from `output/06_clear_results/cfd_resistance_summary.csv`.

`R_CFD` represents the actual pressure-flow behaviour of the full 3D vessel over the inlet-to-outlet segment.


In [ ]:
RESISTANCE_ANALYSIS_DIR = OUTPUT_DIR / "resistance_analysis"
RESISTANCE_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

cfd_resistance_path = OUTPUT_DIR / "cfd_resistance_summary.csv"
if not cfd_resistance_path.exists():
    raise FileNotFoundError(
        f"Missing {cfd_resistance_path}. Run Section 2 first so CFD resistance is available."
    )

cfd_resistance_data = read_csv_rows(cfd_resistance_path)[0]
q_inlet_ml_s = float(cfd_resistance_data["Q_inlet [mL/s]"])
q_inlet_m3_s = q_inlet_ml_s / 1e6
delta_p_inlet_outlet_mmhg = float(cfd_resistance_data["Delta P inlet-outlet [mmHg]"])
delta_p_inlet_outlet_pa = delta_p_inlet_outlet_mmhg * PA_PER_MMHG
r_cfd_pa_s_m3 = float(cfd_resistance_data["R_CFD [Pa·s/m³]"])
r_cfd_mmhg_s_ml = float(cfd_resistance_data["R_CFD [mmHg·s/mL]"])

cfd_resistance_summary_rows = [
    {"Quantity": "Q_inlet", "Value": q_inlet_ml_s, "Units": "mL/s"},
    {"Quantity": "DeltaP_inlet_outlet", "Value": delta_p_inlet_outlet_mmhg, "Units": "mmHg"},
    {"Quantity": "DeltaP_inlet_outlet", "Value": delta_p_inlet_outlet_pa, "Units": "Pa"},
    {"Quantity": "R_CFD", "Value": r_cfd_pa_s_m3, "Units": "Pa·s/m³"},
    {"Quantity": "R_CFD", "Value": r_cfd_mmhg_s_ml, "Units": "mmHg·s/mL"},
]
write_csv(RESISTANCE_ANALYSIS_DIR / "cfd_resistance_loaded.csv", cfd_resistance_summary_rows)
show_table(cfd_resistance_summary_rows, title="Loaded CFD Resistance Summary")

display(Markdown(
    "**Interpretation:** `R_CFD` is the effective hydraulic resistance implied by the 3D CFD pressure drop and inlet flow rate. "
    "It is the pressure-flow behaviour that later reduced-order models should reproduce."
))


## Step 3.2 — Recover Geometric Parameters

The notebook automatically searches repository CSV outputs for geometry-like files containing centerline distance, segment length, radius, diameter, or area information. These sources are used only to recover geometric parameters, not to compare against 0D/1D simulation results.


In [ ]:
def numeric_values(rows, column):
    values = []
    for row in rows:
        try:
            value = float(row[column])
        except (KeyError, TypeError, ValueError):
            continue
        if math.isfinite(value):
            values.append(value)
    return values

def geometry_candidate_from_csv(path):
    try:
        rows = read_csv_rows(path)
    except Exception:
        return None
    if not rows:
        return None

    headers = set(rows[0].keys())
    lower_path = str(path.relative_to(PROJECT_ROOT)).lower()
    radius_column = None
    radius_values_m = []
    for column in ["r_m", "radius_m", "radius_mean_m", "mean_radius_m"]:
        if column in headers:
            vals = numeric_values(rows, column)
            if vals:
                radius_column = column
                radius_values_m = vals
                break
    if not radius_values_m:
        for column in ["diameter_m", "diameter_mean_m"]:
            if column in headers:
                vals = numeric_values(rows, column)
                if vals:
                    radius_column = column
                    radius_values_m = [value / 2.0 for value in vals]
                    break
    if not radius_values_m and {"mean_mm", "min_mm", "max_mm"}.issubset(headers):
        ce_rows = [row for row in rows if row.get("estimator") == "ce_radius"] or rows[:1]
        row = ce_rows[0]
        radius_column = "ce_radius summary"
        radius_values_m = [float(row["mean_mm"]) / 1000.0, float(row["min_mm"]) / 1000.0, float(row["max_mm"]) / 1000.0]

    length_m = None
    length_source = None
    for column in ["segment_length_m", "length_m"]:
        if column in headers:
            vals = numeric_values(rows, column)
            if vals:
                length_m = sum(vals)
                length_source = f"sum({column})"
                break
    if length_m is None and "s_m" in headers:
        vals = numeric_values(rows, "s_m")
        if len(vals) >= 2:
            length_m = max(vals) - min(vals)
            length_source = "max(s_m) - min(s_m)"
    if length_m is None and {"quantity", "value"}.issubset(headers):
        for row in rows:
            quantity = row.get("quantity", "").lower()
            units = row.get("units", row.get("unit", "")).lower()
            if "arc length" in quantity or quantity in {"length", "vessel length"}:
                try:
                    raw_value = float(row["value"])
                except ValueError:
                    continue
                length_m = raw_value / 1000.0 if "mm" in units else raw_value
                length_source = row.get("quantity", "geometry summary")
                break

    if not radius_values_m or length_m is None:
        return None

    score = 0
    if "centerline" in lower_path or "geometry" in lower_path:
        score += 5
    if length_source and length_source.startswith("sum("):
        score += 5
    if "firstblood_segment_parameters" in lower_path:
        score += 12
    if "segment_0d_native_segments" in lower_path:
        score += 10
    if "segment_1d_geometry_profile" in lower_path:
        score += 8
    if "reduced_order_models" in lower_path:
        score += 3
    if len(radius_values_m) > 20:
        score += 2
    if length_m > 0:
        score += 1

    return {
        "path": str(path.relative_to(PROJECT_ROOT)),
        "score": score,
        "length_m": length_m,
        "length_source": length_source,
        "radius_column": radius_column,
        "mean_radius_m": float(np.mean(radius_values_m)),
        "min_radius_m": float(np.min(radius_values_m)),
        "max_radius_m": float(np.max(radius_values_m)),
        "n_radius_values": len(radius_values_m),
    }

def find_geometry_candidates(root):
    candidates = []
    for base in [root / "output", root / "data", root / "openfoam"]:
        if not base.exists():
            continue
        for path in base.rglob("*.csv"):
            if RESISTANCE_ANALYSIS_DIR in path.parents:
                continue
            candidate = geometry_candidate_from_csv(path)
            if candidate:
                candidates.append(candidate)
    return sorted(candidates, key=lambda row: (row["score"], row["n_radius_values"]), reverse=True)

geometry_candidates = find_geometry_candidates(PROJECT_ROOT)
if not geometry_candidates:
    raise FileNotFoundError("No geometry CSV with usable length and radius information was found.")

selected_geometry = geometry_candidates[0]
geometry_source_rows = [
    {
        "source_file": row["path"],
        "score": row["score"],
        "length_source": row["length_source"],
        "radius_source": row["radius_column"],
        "n_radius_values": row["n_radius_values"],
    }
    for row in geometry_candidates[:8]
]
write_csv(RESISTANCE_ANALYSIS_DIR / "geometry_source_candidates.csv", geometry_source_rows)
show_table(geometry_source_rows, title="Geometry Source Candidates")

geometric_parameter_rows = [
    {"Quantity": "Length", "Value": selected_geometry["length_m"], "Units": "m", "Source": selected_geometry["path"]},
    {"Quantity": "Length", "Value": selected_geometry["length_m"] * 1000.0, "Units": "mm", "Source": selected_geometry["path"]},
    {"Quantity": "Mean Radius", "Value": selected_geometry["mean_radius_m"], "Units": "m", "Source": selected_geometry["path"]},
    {"Quantity": "Mean Radius", "Value": selected_geometry["mean_radius_m"] * 1000.0, "Units": "mm", "Source": selected_geometry["path"]},
    {"Quantity": "Minimum Radius", "Value": selected_geometry["min_radius_m"], "Units": "m", "Source": selected_geometry["path"]},
    {"Quantity": "Minimum Radius", "Value": selected_geometry["min_radius_m"] * 1000.0, "Units": "mm", "Source": selected_geometry["path"]},
    {"Quantity": "Maximum Radius", "Value": selected_geometry["max_radius_m"], "Units": "m", "Source": selected_geometry["path"]},
    {"Quantity": "Maximum Radius", "Value": selected_geometry["max_radius_m"] * 1000.0, "Units": "mm", "Source": selected_geometry["path"]},
]
write_csv(RESISTANCE_ANALYSIS_DIR / "geometric_parameters.csv", geometric_parameter_rows)
show_table(
    geometric_parameter_rows,
    columns=["Quantity", "Value", "Units"],
    title="Recovered Geometric Parameters",
)
print(f"Selected geometry source: {selected_geometry['path']}")


## Step 3.3 — Compute Poiseuille Resistance

For an ideal straight circular tube, Poiseuille resistance is:

`R_Poiseuille = 8 μ L / (π r⁴)`

This section uses `μ = 0.004 Pa·s` and computes resistance using the recovered mean, minimum, and maximum radius. The comparison demonstrates the strong `r⁻⁴` sensitivity of hydraulic resistance.


In [ ]:
MU_BLOOD = 0.004
length_m = selected_geometry["length_m"]
radius_cases = [
    ("Mean radius", selected_geometry["mean_radius_m"]),
    ("Minimum radius", selected_geometry["min_radius_m"]),
    ("Maximum radius", selected_geometry["max_radius_m"]),
]

poiseuille_rows = []
for method, radius_m in radius_cases:
    r_poiseuille = 8.0 * MU_BLOOD * length_m / (math.pi * radius_m**4)
    poiseuille_rows.append({
        "Method": method,
        "Radius Used [m]": radius_m,
        "Radius Used [mm]": radius_m * 1000.0,
        "R_Poiseuille [Pa·s/m³]": r_poiseuille,
        "R_Poiseuille [mmHg·s/mL]": r_poiseuille / (PA_PER_MMHG * 1e6),
    })

write_csv(RESISTANCE_ANALYSIS_DIR / "poiseuille_resistance_summary.csv", poiseuille_rows)
show_table(poiseuille_rows, title="Poiseuille Resistance Estimates")

display(Markdown(
    "**Interpretation:** Poiseuille resistance changes strongly with radius because `R ∝ r⁻⁴`. "
    "The minimum-radius estimate is therefore much larger than the maximum-radius estimate, even though both come from the same vessel geometry."
))


## Step 3.4 — Compare CFD Resistance to Poiseuille Resistance

The comparison uses the mean-radius Poiseuille value as the single-radius analytical baseline.

`Difference (%) = 100 × (R_CFD − R_Poiseuille) / R_Poiseuille`

`Correction Factor = R_CFD / R_Poiseuille`


In [ ]:
r_poiseuille_mean = next(row["R_Poiseuille [Pa·s/m³]"] for row in poiseuille_rows if row["Method"] == "Mean radius")
difference_pct = 100.0 * (r_cfd_pa_s_m3 - r_poiseuille_mean) / r_poiseuille_mean
correction_factor = r_cfd_pa_s_m3 / r_poiseuille_mean

resistance_comparison_rows = [
    {"Quantity": "R_CFD [Pa·s/m³]", "Value": r_cfd_pa_s_m3},
    {"Quantity": "R_Poiseuille(mean radius) [Pa·s/m³]", "Value": r_poiseuille_mean},
    {"Quantity": "Difference [%]", "Value": difference_pct},
    {"Quantity": "Correction Factor [-]", "Value": correction_factor},
]
write_csv(RESISTANCE_ANALYSIS_DIR / "cfd_vs_poiseuille_comparison.csv", resistance_comparison_rows)
show_table(resistance_comparison_rows, title="CFD Resistance vs Mean-Radius Poiseuille Resistance")

fig, ax = plt.subplots(figsize=(6.4, 4.2))
labels = ["R_CFD", "R_Poiseuille"]
values = [r_cfd_pa_s_m3, r_poiseuille_mean]
bars = ax.bar(labels, values)
ax.set_ylabel("Resistance [Pa·s/m³]")
ax.set_title("CFD vs Poiseuille Resistance")
for bar, value in zip(bars, values):
    ax.annotate(f"{value:.3g}", (bar.get_x() + bar.get_width() / 2, value), textcoords="offset points", xytext=(0, 6), ha="center")
fig.tight_layout()
fig.savefig(RESISTANCE_ANALYSIS_DIR / "cfd_vs_poiseuille_resistance_bar.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(5.8, 4.0))
ax.bar(["R_CFD / R_Poiseuille"], [correction_factor])
ax.axhline(1.0, linestyle="--", linewidth=1.0)
ax.set_ylabel("Correction factor [-]")
ax.set_title("Poiseuille Correction Factor")
ax.annotate(f"{correction_factor:.3f}", (0, correction_factor), textcoords="offset points", xytext=(0, 6), ha="center")
fig.tight_layout()
fig.savefig(RESISTANCE_ANALYSIS_DIR / "poiseuille_correction_factor.png", bbox_inches="tight")
plt.show()


## Step 3.5 — First Quantification of Information Loss

The difference between CFD resistance and Poiseuille resistance is a quantitative measure of the information lost when reducing the vessel to a simplified hydraulic representation.

If `R_CFD > R_Poiseuille`, the real 3D vessel contains losses that are not represented by ideal Poiseuille theory. Possible causes include curvature, tortuosity, skewed velocity profiles, local acceleration, entrance effects, and secondary flow structures.

If `R_CFD ≈ R_Poiseuille`, most of the vessel behaviour is captured by simple reduced-order resistance.


In [ ]:
if r_cfd_pa_s_m3 > r_poiseuille_mean:
    information_loss_interpretation = (
        "R_CFD is greater than the mean-radius Poiseuille estimate, indicating additional 3D losses not represented by ideal straight-tube theory. "
        "Likely contributors include curvature, tortuosity, skewed velocity profiles, local acceleration, entrance effects, and secondary flow structures."
    )
elif math.isclose(r_cfd_pa_s_m3, r_poiseuille_mean, rel_tol=0.05):
    information_loss_interpretation = (
        "R_CFD is close to the mean-radius Poiseuille estimate, suggesting that this vessel's global resistance is mostly captured by simple reduced-order resistance."
    )
else:
    information_loss_interpretation = (
        "R_CFD is lower than the mean-radius Poiseuille estimate. This can occur when the chosen single-radius baseline is too restrictive or does not represent the effective hydraulic radius of the 3D vessel."
    )

information_loss_rows = [
    {"Quantity": "R_CFD / R_Poiseuille(mean)", "Value": correction_factor},
    {"Quantity": "Difference [%]", "Value": difference_pct},
    {"Quantity": "Information-loss interpretation", "Value": information_loss_interpretation},
]
write_csv(RESISTANCE_ANALYSIS_DIR / "information_loss_from_resistance.csv", information_loss_rows)
show_table(information_loss_rows, title="First Quantification of Information Loss")

display(Markdown("**Interpretation:** " + information_loss_interpretation))


## Step 3.6 — Calibration Metrics for Reduced-Order Models

Define:

`CorrectionFactor = R_CFD / R_Poiseuille`

`ResistanceError = |R_model − R_CFD| / R_CFD × 100`

Future 0D and 1D models should attempt to reproduce `R_CFD` rather than the ideal Poiseuille resistance. This becomes the primary calibration target.


In [ ]:
calibration_metric_rows = [
    {
        "Metric": "CorrectionFactor",
        "Definition": "R_CFD / R_Poiseuille",
        "Value": correction_factor,
        "Use": "Multiplier that maps the ideal mean-radius Poiseuille estimate to the CFD-equivalent resistance.",
    },
    {
        "Metric": "ResistanceError",
        "Definition": "|R_model − R_CFD| / R_CFD × 100",
        "Value": "computed later for each ROM",
        "Use": "Primary error metric for future 0D/1D calibration against CFD resistance.",
    },
    {
        "Metric": "Primary calibration target",
        "Definition": "R_model → R_CFD",
        "Value": r_cfd_pa_s_m3,
        "Use": "Future reduced-order models should reproduce this CFD-equivalent resistance.",
    },
]
write_csv(RESISTANCE_ANALYSIS_DIR / "rom_calibration_metrics.csv", calibration_metric_rows)
show_table(calibration_metric_rows, title="Calibration Metrics for Reduced-Order Models")


## Step 3.7 — Connection to 0D Models

No 0D simulations are loaded or compared here.

A 0D model uses:

`ΔP = R × Q`

Therefore, `R_CFD` can be inserted directly as a lumped resistance.

The conceptual workflow is:

`CFD → Q and ΔP → R_CFD → 0D resistance`


In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 2.1))
ax.axis("off")
workflow_nodes = ["CFD", "Q and ΔP", "R_CFD", "0D resistance"]
x_positions = np.linspace(0.08, 0.92, len(workflow_nodes))
for x, label in zip(x_positions, workflow_nodes):
    ax.text(x, 0.5, label, ha="center", va="center", bbox={"boxstyle": "round,pad=0.35", "fc": "white", "ec": "0.35"})
for x0, x1 in zip(x_positions[:-1], x_positions[1:]):
    ax.annotate("", xy=(x1 - 0.07, 0.5), xytext=(x0 + 0.07, 0.5), arrowprops={"arrowstyle": "->", "linewidth": 1.4})
ax.set_title("CFD Resistance Transfer to a Lumped 0D Model")
fig.tight_layout()
fig.savefig(RESISTANCE_ANALYSIS_DIR / "cfd_to_0d_resistance_workflow.png", bbox_inches="tight")
plt.show()


## Step 3.8 — Connection to 1D Models

No 1D simulation comparison is performed here.

1D models compute resistance from geometry. The correction factor:

`R_CFD / R_Poiseuille`

can later be used to improve 1D predictions by calibrating the geometric/friction-based resistance so that the global pressure-flow response matches the CFD reference.


In [ ]:
section3_conclusion_rows = [
    {"Question": "What is compared in Section 3?", "Conclusion": "Only CFD resistance and analytical Poiseuille resistance are compared."},
    {"Question": "What is the correction factor?", "Conclusion": f"R_CFD / R_Poiseuille(mean radius) = {correction_factor:.6g}."},
    {"Question": "What does the difference mean?", "Conclusion": "It quantifies resistance information lost when a 3D vessel is reduced to an idealized hydraulic representation."},
    {"Question": "What is the calibration target?", "Conclusion": "Future 0D/1D models should reproduce R_CFD rather than the ideal Poiseuille resistance."},
]
write_csv(RESISTANCE_ANALYSIS_DIR / "section3_resistance_analysis_conclusions.csv", section3_conclusion_rows)
show_table(section3_conclusion_rows, title="Section 3 Resistance Analysis Conclusions")
print(f"Section 3 outputs written to: {RESISTANCE_ANALYSIS_DIR}")
